# 11 Presentation Folium Map

This notebook builds a presentation-ready Folium HTML map for the electric pole analysis.

It includes:
- original TIFF RGB layer when available
- CHMv1, CHMv2 minimal, and CHMv2 standardized TIFF outputs as toggleable layers
- manually labeled pole ground truth markers
- hover tooltips and Google Maps links in popups
- min/max canopy-height markers for each model
- an embedded analysis panel with metrics and confusion matrices

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q folium branca rasterio matplotlib seaborn pandas numpy

In [ ]:
from pathlib import Path
import base64
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import folium
from folium import FeatureGroup, LayerControl
from folium.raster_layers import ImageOverlay
from branca.element import Element

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform, transform_bounds
from rasterio.transform import xy
from matplotlib import cm, colors


In [ ]:
csv_path = Path('/content/drive/MyDrive/electric-pole-analysis/reference/matched_rows_only_pole_vegetation_streetview.csv')

chmv1_path = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/chmv1_rgb3_alpha_nodata255_res060_chm.tif')
chmv2_min_path = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/chmv2_minimal_tile512_ov64.tif')
chmv2_std_path = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/chmv2_standardized_tile512_ov64.tif')

robust_results_path = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/pole_filtering/poles_inside_all_three_models_robust_rule_results.csv')
filtered_poles_path = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/pole_filtering/poles_inside_all_three_models.csv')

rgb_native_path = Path('/content/drive/MyDrive/electric-pole-analysi/inputs/rgb3_alpha_native.tif')
rgb_std_path = Path('/content/drive/MyDrive/electric-pole-analysi/inputs/rgb3_alpha_nodata255_res060.tif')

original_candidates = [
    Path('/content/drive/MyDrive/electric-pole-analysi/inputs/IMG_PHR1A_PMS_202508271555298_ORT_b24bf281-b6fd-48dc-c86f-f97e23fc09aa_R1C1.TIF'),
    Path('/content/drive/MyDrive/electric-pole-analysis/inputs/IMG_PHR1A_PMS_202508271555298_ORT_b24bf281-b6fd-48dc-c86f-f97e23fc09aa_R1C1.TIF'),
]
original_tiff_path = next((p for p in original_candidates if p.exists()), None)

output_dir = Path('/content/drive/MyDrive/electric-pole-analysis/outputs/folium')
output_dir.mkdir(parents=True, exist_ok=True)
html_path = output_dir / 'pole_presentation_map.html'

for p in [csv_path, chmv1_path, chmv2_min_path, chmv2_std_path, rgb_native_path, rgb_std_path]:
    print(p.name, '->', p.exists())
print('Robust results CSV ->', robust_results_path.exists())
print('Filtered poles CSV ->', filtered_poles_path.exists())
print('Original TIFF ->', original_tiff_path)
print('Output HTML ->', html_path)

In [ ]:
def classify_manual_risk(row):
    veg_id = row['veg_id']
    veg_name = str(row['veg_name']).strip().lower()
    if veg_id == 0 or veg_name in {'clear', 'safe'}:
        return 'safe'
    if veg_id in {1, 2, 3} or veg_name in {'high', 'watch', 'critical', 'risk'}:
        return 'risk'
    return 'unknown'

poles_df = pd.read_csv(csv_path)
poles_df['manual_risk'] = poles_df.apply(classify_manual_risk, axis=1)

if robust_results_path.exists():
    presentation_df = pd.read_csv(robust_results_path)
    source_name = 'robust results CSV'
elif filtered_poles_path.exists():
    presentation_df = pd.read_csv(filtered_poles_path)
    source_name = 'filtered poles CSV'
else:
    presentation_df = poles_df.copy()
    source_name = 'original pole CSV'

if 'manual_risk' not in presentation_df.columns:
    presentation_df['manual_risk'] = presentation_df.apply(classify_manual_risk, axis=1)

presentation_df['google_maps_url'] = presentation_df.apply(
    lambda r: f"https://www.google.com/maps?q={r['pole_lat']},{r['pole_lng']}",
    axis=1,
)

print('Using pole dataset from:', source_name)
print('Rows:', len(presentation_df))
presentation_df[['pole_id', 'veg_id', 'veg_name', 'manual_risk', 'pole_lat', 'pole_lng']].head()

In [ ]:
def render_chm_overlay(tif_path, layer_name, cmap_name='viridis', vmin=0, vmax=40, max_dim=1200):
    with rasterio.open(tif_path) as src:
        dst_crs = 'EPSG:4326'
        transform_4326, width_4326, height_4326 = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )

        scale = min(1.0, max_dim / max(width_4326, height_4326))
        width_out = max(1, int(width_4326 * scale))
        height_out = max(1, int(height_4326 * scale))

        arr = src.read(1)
        arr_min = float(np.nanmin(arr))
        arr_max = float(np.nanmax(arr))

        min_rc = np.argwhere(np.isfinite(arr) & (arr == arr_min))[0]
        max_rc = np.argwhere(np.isfinite(arr) & (arr == arr_max))[0]
        min_x, min_y = xy(src.transform, int(min_rc[0]), int(min_rc[1]))
        max_x, max_y = xy(src.transform, int(max_rc[0]), int(max_rc[1]))
        min_lon, min_lat = transform(src.crs, dst_crs, [min_x], [min_y])
        max_lon, max_lat = transform(src.crs, dst_crs, [max_x], [max_y])

        dst = np.empty((height_out, width_out), dtype=np.float32)
        dst_transform = rasterio.transform.from_bounds(
            *transform_bounds(src.crs, dst_crs, *src.bounds), width_out, height_out
        )

        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )

        norm = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
        rgba = cm.get_cmap(cmap_name)(norm(np.nan_to_num(dst, nan=vmin)))
        rgba[..., 3] = np.where(np.isfinite(dst), 0.75, 0.0)
        rgba_uint8 = (rgba * 255).astype(np.uint8)

        west, south, east, north = transform_bounds(src.crs, dst_crs, *src.bounds)
        bounds = [[south, west], [north, east]]

        return {
            'name': layer_name,
            'rgba': rgba_uint8,
            'bounds': bounds,
            'min_val': arr_min,
            'max_val': arr_max,
            'min_lat': float(min_lat[0]),
            'min_lng': float(min_lon[0]),
            'max_lat': float(max_lat[0]),
            'max_lng': float(max_lon[0]),
        }

def render_rgb_overlay(tif_path, layer_name, max_dim=1400):
    with rasterio.open(tif_path) as src:
        dst_crs = 'EPSG:4326'
        transform_4326, width_4326, height_4326 = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        scale = min(1.0, max_dim / max(width_4326, height_4326))
        width_out = max(1, int(width_4326 * scale))
        height_out = max(1, int(height_4326 * scale))

        rgb = src.read([1, 2, 3]).astype(np.float32)
        rgb_out = np.empty((3, height_out, width_out), dtype=np.float32)
        dst_transform = rasterio.transform.from_bounds(
            *transform_bounds(src.crs, dst_crs, *src.bounds), width_out, height_out
        )

        for i in range(3):
            reproject(
                source=rgb[i],
                destination=rgb_out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
            )

        rgb_hwc = np.moveaxis(rgb_out, 0, -1)
        rgb_disp = rgb_hwc.copy()
        for b in range(3):
            vals = rgb_disp[..., b]
            valid = vals > 0
            if valid.any():
                p2, p98 = np.percentile(vals[valid], (2, 98))
                if p98 > p2:
                    rgb_disp[..., b] = np.clip((vals - p2) / (p98 - p2), 0, 1)

        alpha = np.where(np.all(rgb_hwc > 0, axis=-1), 0.9, 0.0)
        rgba = np.dstack([rgb_disp, alpha])
        rgba_uint8 = (rgba * 255).astype(np.uint8)

        west, south, east, north = transform_bounds(src.crs, dst_crs, *src.bounds)
        bounds = [[south, west], [north, east]]
        return {'name': layer_name, 'rgba': rgba_uint8, 'bounds': bounds}

layers = [
    render_chm_overlay(chmv1_path, 'CHMv1'),
    render_chm_overlay(chmv2_min_path, 'CHMv2 minimal'),
    render_chm_overlay(chmv2_std_path, 'CHMv2 standardized'),
]

rgb_layer = render_rgb_overlay(original_tiff_path, 'Original TIFF RGB') if original_tiff_path else None
[(layer['name'], round(layer['min_val'], 2), round(layer['max_val'], 2)) for layer in layers]

In [ ]:
if robust_results_path.exists() and {'manual_risk', 'chmv1_risk_robust', 'chmv2_min_risk_robust', 'chmv2_std_risk_robust'}.issubset(set(presentation_df.columns)):
    eval_df = presentation_df[presentation_df['manual_risk'].isin({'risk', 'safe'})].copy()

    def make_confusion_table(df_in, pred_col):
        cm_df = pd.crosstab(df_in['manual_risk'], df_in[pred_col], rownames=['Manual'], colnames=['Predicted'])
        return cm_df.reindex(index=['risk', 'safe'], columns=['risk', 'safe'], fill_value=0)

    def classification_metrics(cm_df):
        tp = cm_df.loc['risk', 'risk']
        fn = cm_df.loc['risk', 'safe']
        fp = cm_df.loc['safe', 'risk']
        tn = cm_df.loc['safe', 'safe']
        total = tp + fn + fp + tn
        accuracy = (tp + tn) / total if total else np.nan
        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else np.nan
        return {'accuracy': accuracy * 100, 'precision': precision * 100, 'recall': recall * 100, 'specificity': specificity * 100, 'f1': f1 * 100, 'n_used': total}

    cm_chmv1 = make_confusion_table(eval_df, 'chmv1_risk_robust')
    cm_chmv2_min = make_confusion_table(eval_df, 'chmv2_min_risk_robust')
    cm_chmv2_std = make_confusion_table(eval_df, 'chmv2_std_risk_robust')

    metrics_pct = pd.DataFrame([
        {'model': 'CHMv1', **classification_metrics(cm_chmv1)},
        {'model': 'CHMv2 minimal', **classification_metrics(cm_chmv2_min)},
        {'model': 'CHMv2 standardized', **classification_metrics(cm_chmv2_std)},
    ])
else:
    metrics_pct = pd.DataFrame([
        {'model': 'CHMv1', 'accuracy': 52.70, 'precision': 78.57, 'recall': 16.92, 'specificity': 94.59, 'f1': 27.85, 'n_used': 241},
        {'model': 'CHMv2 minimal', 'accuracy': 72.20, 'precision': 81.82, 'recall': 62.31, 'specificity': 83.78, 'f1': 70.74, 'n_used': 241},
        {'model': 'CHMv2 standardized', 'accuracy': 69.71, 'precision': 81.32, 'recall': 56.92, 'specificity': 84.68, 'f1': 66.97, 'n_used': 241},
    ])
    cm_chmv1 = pd.DataFrame([[22, 108], [6, 105]], index=['risk', 'safe'], columns=['risk', 'safe'])
    cm_chmv2_min = pd.DataFrame([[81, 49], [18, 93]], index=['risk', 'safe'], columns=['risk', 'safe'])
    cm_chmv2_std = pd.DataFrame([[74, 56], [17, 94]], index=['risk', 'safe'], columns=['risk', 'safe'])

metrics_pct

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
items = [
    ('CHMv1', cm_chmv1),
    ('CHMv2 minimal', cm_chmv2_min),
    ('CHMv2 standardized', cm_chmv2_std),
]

for ax, (title, cm_df) in zip(axes, items):
    sns.heatmap(
        cm_df,
        annot=True,
        fmt='d',
        cmap='YlGnBu',
        cbar=False,
        square=True,
        linewidths=1,
        linecolor='white',
        annot_kws={'size': 14, 'weight': 'bold'},
        ax=ax,
    )
    row = metrics_pct.loc[metrics_pct['model'] == title].iloc[0]
    ax.set_title(
        f"{title}\nAcc {row['accuracy']:.2f}% | Prec {row['precision']:.2f}% | Recall {row['recall']:.2f}% | F1 {row['f1']:.2f}%",
        fontsize=12,
        weight='bold',
    )
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('Manual label')

plt.suptitle('Robust Rule Confusion Matrices', fontsize=18, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=180, bbox_inches='tight')
plt.close(fig)
confusion_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
confusion_img_html = f"<img src='data:image/png;base64,{confusion_b64}' style='width:100%; border:1px solid #ccc; border-radius:6px;'/>"

In [ ]:
map_center = [presentation_df['pole_lat'].mean(), presentation_df['pole_lng'].mean()]
m = folium.Map(location=map_center, zoom_start=15, tiles='OpenStreetMap', control_scale=True)
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)

if rgb_layer is not None:
    fg_rgb = FeatureGroup(name='Original TIFF RGB', show=True)
    ImageOverlay(
        image=rgb_layer['rgba'],
        bounds=rgb_layer['bounds'],
        opacity=0.85,
        interactive=False,
        cross_origin=False,
        zindex=1,
    ).add_to(fg_rgb)
    fg_rgb.add_to(m)

for layer in layers:
    fg = FeatureGroup(name=f"{layer['name']} CHM", show=(layer['name'] == 'CHMv2 minimal'))
    ImageOverlay(
        image=layer['rgba'],
        bounds=layer['bounds'],
        opacity=0.72,
        interactive=False,
        cross_origin=False,
        zindex=2,
    ).add_to(fg)

    folium.CircleMarker(
        location=[layer['min_lat'], layer['min_lng']],
        radius=5,
        color='cyan',
        fill=True,
        fill_opacity=1,
        tooltip=f"{layer['name']} min canopy: {layer['min_val']:.2f} m",
        popup=folium.Popup(
            f"<b>{layer['name']} min</b><br>Height: {layer['min_val']:.2f} m<br>Lat: {layer['min_lat']:.6f}<br>Lng: {layer['min_lng']:.6f}",
            max_width=280,
        ),
    ).add_to(fg)

    folium.CircleMarker(
        location=[layer['max_lat'], layer['max_lng']],
        radius=5,
        color='red',
        fill=True,
        fill_opacity=1,
        tooltip=f"{layer['name']} max canopy: {layer['max_val']:.2f} m",
        popup=folium.Popup(
            f"<b>{layer['name']} max</b><br>Height: {layer['max_val']:.2f} m<br>Lat: {layer['max_lat']:.6f}<br>Lng: {layer['max_lng']:.6f}",
            max_width=280,
        ),
    ).add_to(fg)

    fg.add_to(m)

safe_fg = FeatureGroup(name='Ground truth poles: safe', show=True)
risk_fg = FeatureGroup(name='Ground truth poles: risk', show=True)
unknown_fg = FeatureGroup(name='Ground truth poles: unknown', show=False)

for _, row in presentation_df.iterrows():
    label = row['manual_risk']
    color = {'safe': 'lime', 'risk': 'red'}.get(label, 'orange')
    target_group = safe_fg if label == 'safe' else risk_fg if label == 'risk' else unknown_fg

    popup_html = f"""
    <b>Pole ID:</b> {row['pole_id']}<br>
    <b>Manual label:</b> {label}<br>
    <b>veg_id:</b> {row['veg_id']}<br>
    <b>veg_name:</b> {row['veg_name']}<br>
    <b>Lat:</b> {row['pole_lat']:.6f}<br>
    <b>Lng:</b> {row['pole_lng']:.6f}<br>
    <a href='{row['google_maps_url']}' target='_blank'>Open in Google Maps</a>
    """
    tooltip_text = f"Pole {row['pole_id']} | {label} | ({row['pole_lat']:.6f}, {row['pole_lng']:.6f})"

    folium.CircleMarker(
        location=[row['pole_lat'], row['pole_lng']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        weight=1,
        tooltip=tooltip_text,
        popup=folium.Popup(popup_html, max_width=320),
    ).add_to(target_group)

safe_fg.add_to(m)
risk_fg.add_to(m)
unknown_fg.add_to(m)
LayerControl(collapsed=False).add_to(m)

metrics_table_html = metrics_pct[['model', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'n_used']].round(2).to_html(index=False)

summary_panel = f"""
<div style="position: fixed; top: 10px; right: 10px; width: 470px; max-height: 90vh; overflow-y: auto; z-index: 9999; background: white; border: 2px solid #444; border-radius: 8px; padding: 12px; box-shadow: 0 2px 12px rgba(0,0,0,0.25); font-size: 13px;">
  <h3 style="margin-top:0;">Electric Pole Analysis Summary</h3>
  <p><b>Ground truth poles:</b> {len(presentation_df)}<br>
  <b>Current best model:</b> CHMv2 minimal<br>
  <b>Rule:</b> robust neighborhood rule from notebook 10</p>
  <p><b>Robust rule:</b> risk if <code>buffer_p95_m &gt;= pole_height_m</code> or <code>buffer_above_pole_pct &gt;= 20%</code></p>
  <h4>Metrics</h4>
  {metrics_table_html}
  <h4>Confusion Matrices</h4>
  {confusion_img_html}
  <h4>How to use this map</h4>
  <ul>
    <li>Toggle the original TIFF and CHM layers in the layer control.</li>
    <li>Hover on a pole to see coordinates.</li>
    <li>Click a pole to open a popup with the Google Maps link.</li>
    <li>Min/max canopy markers are shown for each model layer.</li>
  </ul>
</div>
"""

m.get_root().html.add_child(Element(summary_panel))
m.save(str(html_path))
print('Saved HTML map to:', html_path)
m